# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MA-1305/Project_Mahin-1305/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

repo_path = "/content/Project_Mahin-1305"

if not os.path.exists(repo_path):
    !git clone https://github.com/MA-1305/Project_Mahin-1305.git /content/Project_Mahin-1305

print("Repository exists:", os.path.exists(repo_path))

Repository exists: True


In [11]:
from pathlib import Path

data_path = Path(
    "/content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv"
)

print("CSV exists:", data_path.exists())
print("CSV path:", data_path)

CSV exists: True
CSV path: /content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv


In [12]:
import pandas as pd
import numpy as np

data_path = "/content/Project_Mahin-1305/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

df["is_declining"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Rows:", len(df))

print("\nKey distributions:")

print(
    df[
        [
            "impressions_90d",
            "sessions_90d",
            "content_age_days",
            "days_since_last_update",
            "ctr",
            "avg_position"
        ]
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    ).T
)

Rows: 30000

Key distributions:
                          count         mean           std   min     50%  \
impressions_90d         30000.0  5200.366300  16838.019547   1.0  731.00   
sessions_90d            30000.0    37.066633    107.069131   1.0    7.00   
content_age_days        30000.0   256.167800    132.707930  90.0  236.00   
days_since_last_update  30000.0    46.098300     42.078709   1.0   20.00   
ctr                     30000.0     0.510733      3.279162   0.0    0.07   
avg_position            30000.0    16.342380     15.216790   0.0   10.80   

                            75%       90%       95%        99%       max  
impressions_90d         3615.25  12136.40  22996.50  73505.830  517715.0  
sessions_90d              27.00     88.00    166.00    451.010    4345.0  
content_age_days         333.00    463.00    487.00    537.000     564.0  
days_since_last_update   104.00    104.00    104.00    106.000     373.0  
ctr                        0.29      0.65      1.09      8.3

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: Staleness
stale = df["days_since_last_update"] >= 180
fresh = df["days_since_last_update"] < 180

stale_rate = df.loc[stale, "is_declining"].mean()
fresh_rate = df.loc[fresh, "is_declining"].mean()


# Signal 2: Search volume
high_volume = df["impressions_90d"] >= 500
low_volume = df["impressions_90d"] < 500

high_volume_rate = df.loc[high_volume, "is_declining"].mean()
low_volume_rate = df.loc[low_volume, "is_declining"].mean()


# Signal 3: CTR relative to position
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

normal_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] >= 0.5)
)

low_ctr_rate = df.loc[low_ctr_visible, "is_declining"].mean()
normal_ctr_rate = df.loc[normal_ctr_visible, "is_declining"].mean()


print("SIGNAL 1 — STALENESS")
print("Stale decline rate:", round(stale_rate, 4))
print("Fresh decline rate:", round(fresh_rate, 4))

print("\nSIGNAL 2 — SEARCH VOLUME")
print("High-volume decline rate:", round(high_volume_rate, 4))
print("Low-volume decline rate:", round(low_volume_rate, 4))

print("\nSIGNAL 3 — CTR RELATIVE TO POSITION")
print("Low-CTR visible decline rate:", round(low_ctr_rate, 4))
print("Normal-CTR visible decline rate:", round(normal_ctr_rate, 4))

SIGNAL 1 — STALENESS
Stale decline rate: 0.4713
Fresh decline rate: 0.5425

SIGNAL 2 — SEARCH VOLUME
High-volume decline rate: 0.5955
Low-volume decline rate: 0.4747

SIGNAL 3 — CTR RELATIVE TO POSITION
Low-CTR visible decline rate: 0.6271
Normal-CTR visible decline rate: 0.4753


Three signals are tested using the observed decline label. Each signal compares the decline rate between two clearly defined groups.

The verdict for each signal should be based on the measured difference, rather than being assumed in advance.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag: visible pages with relatively low CTR
flagged = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

not_flagged = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] >= 0.5)
)

flagged_rate = df.loc[flagged, "is_declining"].mean()
not_flagged_rate = df.loc[not_flagged, "is_declining"].mean()

print("Flagged pages:", flagged.sum())
print("Not-flagged visible pages:", not_flagged.sum())

print("\nDecline rate among flagged pages:",
      round(flagged_rate, 4))

print("Decline rate among not-flagged visible pages:",
      round(not_flagged_rate, 4))

print("\nDifference:",
      round(flagged_rate - not_flagged_rate, 4))

Flagged pages: 9759
Not-flagged visible pages: 2264

Decline rate among flagged pages: 0.6271
Decline rate among not-flagged visible pages: 0.4753

Difference: 0.1518


Flag tested: low CTR among visible pages.

The rule assumes that pages receiving meaningful impressions and ranking within the top 20 positions but having relatively low CTR may deserve review. I compare the observed decline rate of flagged pages with similar visible pages that do not meet the low-CTR condition.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The signal audit suggests that content teams should use multiple observable signals together when prioritizing pages for review rather than relying on one flag alone. Pages with stronger evidence of opportunity should be reviewed first, while low-volume or ambiguous cases should be treated more cautiously. These signals support decision-making but do not prove that a content refresh will cause improved performance.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Practical takeaway:")
print("Use multiple observable signals to prioritize pages for human review.")
print("Treat low-volume or ambiguous cases cautiously.")
print("Model and signal results are decision-support, not proof of causation.")

Practical takeaway:
Use multiple observable signals to prioritize pages for human review.
Treat low-volume or ambiguous cases cautiously.
Model and signal results are decision-support, not proof of causation.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.